In [1]:
# -----------------------------
# Install dependencies
# -----------------------------
!pip install transformers accelerate datasets peft -q

In [3]:
# -----------------------------
# Imports
# -----------------------------
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, Trainer, TrainingArguments
from peft import LoraConfig, get_peft_model


In [4]:
# -----------------------------
# Synthetic train_data (300 examples)
# -----------------------------
import random

instructions = [
    "Customer asks about refund window",
    "Customer wants to cancel an order",
    "Order arrived late",
    "Wrong item received",
    "Product not working",
    "Shipping cost inquiry",
    "Change delivery address",
    "Request for invoice",
    "Ask about warranty",
    "Technical support request"
]

responses = [
    "Our refund window is 30 days from delivery.",
    "You can cancel your order from your account dashboard within 24 hours.",
    "Sorry for the delay. A delivery credit has been applied.",
    "We’ll ship the correct item and provide a return label.",
    "Please try resetting the product. Contact support if the issue persists.",
    "Shipping cost depends on your location and chosen delivery speed.",
    "You can update your delivery address before the order ships.",
    "An invoice will be emailed to you after purchase.",
    "Your product comes with a 12-month warranty.",
    "Our tech support team will contact you shortly."
]

train_data = []
for i in range(300):
    idx = random.randint(0, len(instructions) - 1)
    train_data.append({
        "instruction": f"{instructions[idx]} #{i+1}",
        "response": responses[idx]
    })

In [8]:
# -----------------------------
# Load tokenizer and model
# -----------------------------
model_name = "microsoft/phi-2"  # 1.1B TinyLlama, GPTQ version
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

device = "cuda" if torch.cuda.is_available() else "cpu"

base_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",
    torch_dtype=torch.float16 if device=="cuda" else torch.float32
)

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

added_tokens.json: 0.00B [00:00, ?B/s]

...

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/5.00G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/564M [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

In [9]:
# -----------------------------
# Preprocess train_data
# -----------------------------
def preprocess(example):
    prompt = f"User: {example['instruction']}\nAssistant: {example['response']}"
    enc = tokenizer(prompt, padding="max_length", truncation=True, max_length=128)
    enc["labels"] = enc["input_ids"].copy()
    return enc

from datasets import Dataset
dataset = Dataset.from_list(train_data)
tokenized_dataset = dataset.map(preprocess)
tokenized_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])

Map:   0%|          | 0/300 [00:00<?, ? examples/s]

In [10]:
# -----------------------------
# LoRA configuration
# -----------------------------
lora_cfg = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj","v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(base_model, lora_cfg)
model.print_trainable_parameters()

trainable params: 2,621,440 || all params: 2,782,305,280 || trainable%: 0.0942


In [ ]:
# -----------------------------
# Training arguments
# -----------------------------
training_args = TrainingArguments(
    output_dir="./tinyllama-lora",
    learning_rate=2e-4,
    per_device_train_batch_size=4,
    num_train_epochs=5,
    logging_steps=5,
    save_strategy="no",
    report_to="none",
    fp16=True if device=="cuda" else False
)

# -----------------------------
# Trainer
# -----------------------------
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset
)

trainer.train()

The model is already on multiple devices. Skipping the move to device specified in `args`.
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


In [ ]:
# -----------------------------
# Inference
# -----------------------------
def generate(prompt):
    text = f"User: {prompt}\nAssistant:"
    inp = tokenizer(text, return_tensors="pt").to(device)
    out = model.generate(
        **inp,
        max_new_tokens=50,
        pad_token_id=tokenizer.eos_token_id,
        do_sample=True,
        temperature=0.7
    )
    return tokenizer.decode(out[0], skip_special_tokens=True)

# -----------------------------
# Test examples
# -----------------------------
for p in [
    "Customer asks about refund window",
    "Order arrived late",
    "Wrong item received",
    "How do I request an invoice?",
    "Product is not working"
]:
    print("----")
    print(generate(p))